In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

warnings.filterwarnings("ignore")
np.random.seed(42)

classification_datasets = [
    "bank_marketing",
    "customer_churn",
    "hr_attrition"
]
regression_datasets = [
    "house_prices",
    "medical_insurance"
]

def load_dataset(dataset_name, base_path="."):
    dataset_path = os.path.join(base_path, dataset_name)

    required_files = [
        "X_train_processed.csv",
        "X_val_processed.csv",
        "X_test_processed.csv",
        "y_train.csv",
        "y_val.csv",
        "y_test.csv"
    ]

    for fname in required_files:
        fpath = os.path.join(dataset_path, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing file: {fpath}")

    X_train = pd.read_csv(os.path.join(dataset_path, "X_train_processed.csv"))
    X_val = pd.read_csv(os.path.join(dataset_path, "X_val_processed.csv"))
    X_test = pd.read_csv(os.path.join(dataset_path, "X_test_processed.csv"))

    y_train = pd.read_csv(os.path.join(dataset_path, "y_train.csv")).squeeze("columns")
    y_val = pd.read_csv(os.path.join(dataset_path, "y_val.csv")).squeeze("columns")
    y_test = pd.read_csv(os.path.join(dataset_path, "y_test.csv")).squeeze("columns")

    return X_train, X_val, X_test, y_train, y_val, y_test

def get_scale_pos_weight(y):
    neg = (y == 0).sum()
    pos = (y == 1).sum()

    if pos == 0:
        return 1.0

    return neg / pos

def run_xgb_classification(dataset_name, params, base_path="."):
    X_train, X_val, X_test, y_train, y_val, y_test = load_dataset(dataset_name, base_path)

    if y_train.nunique() != 2:
        raise ValueError(f"{dataset_name} is not a binary classification dataset.")
    
    scale_pos_weight = get_scale_pos_weight(y_train)
    
    clf_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        early_stopping_rounds=30,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        **params
    )

    start_train = time.time()
    clf_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    train_time = time.time() - start_train

    val_pred = clf_model.predict(X_val)
    val_prob = clf_model.predict_proba(X_val)[:, 1]

    result = {
        "dataset": dataset_name,
        "task": "classification",
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_auc": roc_auc_score(y_val, val_prob),
        "train_time_sec": train_time,
        "params": params,
        "best_iteration": clf_model.best_iteration
    }

    return clf_model, result

def run_xgb_regression(dataset_name, params, base_path="."):
    X_train, X_val, X_test, y_train, y_val, y_test = load_dataset(dataset_name, base_path)

    reg_model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
        **params
    )

    start_train = time.time()
    reg_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    train_time = time.time() - start_train

    val_pred = reg_model.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    val_mae = mean_absolute_error(y_val, val_pred)
    val_r2 = r2_score(y_val, val_pred)

    result = {
        "dataset": dataset_name,
        "task": "regression",
        "val_rmse": val_rmse,
        "val_mae": val_mae,
        "val_r2": val_r2,
        "train_time_sec": train_time,
        "params": params,
        "best_iteration": reg_model.best_iteration
    }

    return reg_model, result


default_clf_params = {
    "n_estimators": 1000,
    "max_depth": 5,
    "learning_rate": 0.05,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1
}
def tune_xgb_classification(dataset_name, param_grid, base_path="."):
    best_clf_score = -np.inf
    best_clf_result = None
    best_clf_model = None
    best_clf_params = None
    best_clf_iteration = None

    for n_estimators in param_grid["n_estimators"]:
        for max_depth in param_grid["max_depth"]:
            for learning_rate in param_grid["learning_rate"]:
                for min_child_weight in param_grid["min_child_weight"]:
                    for subsample in param_grid["subsample"]:
                        for colsample_bytree in param_grid["colsample_bytree"]:
                            for reg_lambda in param_grid["reg_lambda"]:

                                params = {
                                    "n_estimators": n_estimators,
                                    "max_depth": max_depth,
                                    "learning_rate": learning_rate,
                                    "min_child_weight": min_child_weight,
                                    "subsample": subsample,
                                    "colsample_bytree": colsample_bytree,
                                    "reg_lambda": reg_lambda
                                }

                                model, result = run_xgb_classification(
                                    dataset_name, params, base_path
                                )

                                score = result["val_auc"]

                                if score > best_clf_score:
                                    best_clf_score = score
                                    best_clf_result = result
                                    best_clf_model = model
                                    best_clf_params = params
                                    best_clf_iteration = result["best_iteration"]

    return best_clf_model, best_clf_result, best_clf_params, best_clf_iteration

default_reg_params = {
    "n_estimators": 1000,
    "max_depth": 5,
    "learning_rate": 0.05,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1
}
def tune_xgb_regression(dataset_name, param_grid, base_path="."):
    best_reg_score = np.inf
    best_reg_result = None
    best_reg_model = None
    best_reg_params = None
    best_reg_iteration = None

    for n_estimators in param_grid["n_estimators"]:
        for max_depth in param_grid["max_depth"]:
            for learning_rate in param_grid["learning_rate"]:
                for min_child_weight in param_grid["min_child_weight"]:
                    for subsample in param_grid["subsample"]:
                        for colsample_bytree in param_grid["colsample_bytree"]:
                            for reg_lambda in param_grid["reg_lambda"]:

                                params = {
                                    "n_estimators": n_estimators,
                                    "max_depth": max_depth,
                                    "learning_rate": learning_rate,
                                    "min_child_weight": min_child_weight,
                                    "subsample": subsample,
                                    "colsample_bytree": colsample_bytree,
                                    "reg_lambda": reg_lambda
                                }

                                model, result = run_xgb_regression(
                                    dataset_name, params, base_path
                                )

                                score = result["val_rmse"]

                                if score < best_reg_score:
                                    best_reg_score = score
                                    best_reg_result = result
                                    best_reg_model = model
                                    best_reg_params = params
                                    best_reg_iteration = result["best_iteration"]

    return best_reg_model, best_reg_result, best_reg_params, best_reg_iteration


def test_xgb_classification(dataset_name, params, best_iteration=None, base_path="."):
    X_train, X_val, X_test, y_train, y_val, y_test = load_dataset(dataset_name, base_path)

    X_train_final = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_train_final = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

    final_params = params.copy()
    if best_iteration is not None:
        final_params["n_estimators"] = best_iteration + 1

    scale_pos_weight = get_scale_pos_weight(y_train_final)

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        **final_params
    )

    start_train = time.time()
    model.fit(X_train_final, y_train_final, verbose=False)
    train_time = time.time() - start_train

    start_infer = time.time()
    test_pred = model.predict(X_test)
    test_prob = model.predict_proba(X_test)[:, 1]
    infer_time_total = time.time() - start_infer
    infer_time_per_sample = (infer_time_total / len(X_test))*1000

    result = {
        "dataset": dataset_name,
        "task": "classification",
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_auc": roc_auc_score(y_test, test_prob),
        "train_time_sec": train_time,
        "infer_time_per_sample_sec": infer_time_per_sample,
        "params": final_params
    }

    return model, result

def test_xgb_regression(dataset_name, params, best_iteration=None, base_path="."):
    X_train, X_val, X_test, y_train, y_val, y_test = load_dataset(dataset_name, base_path)

    X_train_final = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_train_final = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

    final_params = params.copy()
    if best_iteration is not None:
        final_params["n_estimators"] = best_iteration + 1

    model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
        **final_params
    )

    start_train = time.time()
    model.fit(X_train_final, y_train_final, verbose=False)
    train_time = time.time() - start_train

    start_infer = time.time()
    test_pred = model.predict(X_test)
    infer_time_total = time.time() - start_infer
    infer_time_per_sample = (infer_time_total / len(X_test))*1000

    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_mae = mean_absolute_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)

    result = {
        "dataset": dataset_name,
        "task": "regression",
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
        "train_time_sec": train_time,
        "infer_time_per_sample_sec": infer_time_per_sample,
        "params": final_params
    }

    return model, result


clf_param_grid = {
    "n_estimators": [1000],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1],
    "min_child_weight": [1, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8],
    "reg_lambda": [1, 5]
}
classification_results = []
classification_models = {}
classification_best_params = {}
for dataset_name in classification_datasets:
    print(f"Tuning classification dataset: {dataset_name}")

    # baseline
    _, baseline_result = run_xgb_classification(
        dataset_name,
        default_clf_params
    )

    # tune on valid
    best_val_model, best_val_result, best_params, best_iteration = tune_xgb_classification(
        dataset_name,
        clf_param_grid
    )

    # final test
    final_model, final_test_result = test_xgb_classification(
        dataset_name,
        best_params,
        best_iteration=best_iteration
    )

    classification_results.append({
        "dataset": dataset_name,
        "model": "XGBoost",
        "task_type": "classification",

        "accuracy": final_test_result["test_accuracy"],
        "auroc": final_test_result["test_auc"],

        "rmse": "",
        "mae": "",
        "r2": "",

        "train_seconds": final_test_result["train_time_sec"],
        "infer_seconds": final_test_result["infer_time_per_sample_sec"],

        "device": "cpu"
    })

    classification_models[dataset_name] = final_model
    classification_best_params[dataset_name] = best_params

    print("Baseline Validation Accuracy:", baseline_result["val_accuracy"])
    print("Baseline Validation AUC:", baseline_result["val_auc"])
    print("Best params:", final_test_result["params"])
    print("Best Validation Accuracy:", best_val_result["val_accuracy"])
    print("Best Validation AUC:", best_val_result["val_auc"])
    print("Final Test Accuracy:", final_test_result["test_accuracy"])
    print("Final Test AUC:", final_test_result["test_auc"])
    print("-" * 50)


reg_param_grid = {
    "n_estimators": [1000],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1],
    "min_child_weight": [1, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8],
    "reg_lambda": [1, 5]
}
regression_results = []
regression_models = {}
regression_best_params = {}
for dataset_name in regression_datasets:
    print(f"Tuning regression dataset: {dataset_name}")

    # baseline
    baseline_model, baseline_result = run_xgb_regression(
        dataset_name,
        default_reg_params
    )

    # tune on valid
    best_val_model, best_val_result, best_params, best_iteration = tune_xgb_regression(
        dataset_name,
        reg_param_grid
    )

    # final test
    final_model, final_test_result = test_xgb_regression(
        dataset_name,
        best_params,
        best_iteration=best_iteration
    )

    regression_results.append({
        "dataset": dataset_name,
        "model": "XGBoost",
        "task_type": "regression",

        "accuracy": "",
        "auroc": "",

        "rmse": final_test_result["test_rmse"],
        "mae": final_test_result["test_mae"],
        "r2": final_test_result["test_r2"],

        "train_seconds": final_test_result["train_time_sec"],
        "infer_seconds": final_test_result["infer_time_per_sample_sec"],

        "device": "cpu"
    })

    regression_models[dataset_name] = final_model
    regression_best_params[dataset_name] = best_params

    print("baseline Validation RMSE:", baseline_result["val_rmse"])
    print("Best params:", final_test_result["params"])
    print("Best Validation RMSE:", best_val_result["val_rmse"])
    print("Best Validation MAE:", best_val_result["val_mae"])
    print("Best Validation R2:", best_val_result["val_r2"])
    print("Final Test RMSE:", final_test_result["test_rmse"])
    print("Final Test MAE:", final_test_result["test_mae"])
    print("Final Test R2:", final_test_result["test_r2"])
    print("-" * 50)


all_results = classification_results + regression_results

columns_order = [
    "dataset", "model", "task_type",
    "accuracy", "auroc",
    "rmse", "mae", "r2",
    "train_seconds", "infer_seconds",
    "device"
]

results_df = pd.DataFrame(all_results)[columns_order]

results_df.to_csv("xgboost_results_summary.csv", index=False)
print("Saved to xgboost_results_summary.csv")


Tuning classification dataset: bank_marketing
Baseline Validation Accuracy: 0.8179003243880861
Baseline Validation AUC: 0.808275450985935
Best params: {'n_estimators': 258, 'max_depth': 7, 'learning_rate': 0.03, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5}
Best Validation Accuracy: 0.8260100265408434
Best Validation AUC: 0.8111739852500972
Final Test Accuracy: 0.8246829843703922
Final Test AUC: 0.7972617726866638
--------------------------------------------------
Tuning classification dataset: customer_churn
Baseline Validation Accuracy: 0.824
Baseline Validation AUC: 0.8886699291665298
Best params: {'n_estimators': 60, 'max_depth': 5, 'learning_rate': 0.1, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5}
Best Validation Accuracy: 0.818
Best Validation AUC: 0.890402447969696
Final Test Accuracy: 0.806
Final Test AUC: 0.8723917964195076
--------------------------------------------------
Tuning classification dataset

In [2]:
results_df

,dataset,model,task_type,accuracy,auroc,rmse,mae,r2,train_seconds,infer_seconds,device
0,bank_marketing,XGBoost,classification,0.824683,0.797262,,,,0.733742,0.001450,cpu
1,customer_churn,XGBoost,classification,0.806,0.872392,,,,0.106605,0.001169,cpu
2,hr_attrition,XGBoost,classification,0.79638,0.819069,,,,0.078657,0.013017,cpu
3,house_prices,XGBoost,regression,,,24159.34966,15255.960938,0.923305,0.610348,0.025265,cpu
4,medical_insurance,XGBoost,regression,,,2904.939062,1756.611957,0.176524,0.515896,0.000338,cpu


In [21]:
def scaling_experiment_xgb_regression(
    dataset_name,
    params,
    sample_sizes=[1000, 2000, 5000, 10000, 20000, 50000, 100000],
    base_path=".",
    random_state=42
):
    X_train, X_val, X_test, y_train, y_val, y_test = load_dataset(dataset_name, base_path)

    X_full = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)

    shuffled_idx = X_full.sample(frac=1, random_state=random_state).index
    X_full = X_full.loc[shuffled_idx].reset_index(drop=True)
    y_full = y_full.loc[shuffled_idx].reset_index(drop=True)

    scaling_results = []

    for n in sample_sizes:
        if n > len(X_full):
            print(f"Skipping n={n}, exceeds full data size ({len(X_full)})")
            continue

        print(f"Running scaling on {dataset_name}, n={n}")

        X_train_sub = X_full.iloc[:n]
        y_train_sub = y_full.iloc[:n]

        model = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
            **params
        )

        start_train = time.time()
        model.fit(X_train_sub, y_train_sub, verbose=False)
        train_time = time.time() - start_train

        test_pred = model.predict(X_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

        scaling_results.append({
            "n": n,
            "test_performance": test_rmse,
            "training_time_s": train_time
        })

    return pd.DataFrame(scaling_results)

medical_insurance_scaling_df = scaling_experiment_xgb_regression(
    dataset_name="medical_insurance",
    params=regression_best_params["medical_insurance"]
)

medical_insurance_scaling_df
# medical_insurance_scaling_df.to_csv("xgboost_medical_insurance_scaling.csv", index=False)
# print("Saved to xgboost_medical_insurance_scaling.csv")

Running scaling on medical_insurance, n=1000
Running scaling on medical_insurance, n=2000
Running scaling on medical_insurance, n=5000
Running scaling on medical_insurance, n=10000
Running scaling on medical_insurance, n=20000
Running scaling on medical_insurance, n=50000
Running scaling on medical_insurance, n=100000


,n,test_performance,training_time_s
0,1000,3102.267765,1.158653
1,2000,2997.914995,1.106956
2,5000,2914.053071,1.116015
3,10000,2876.886617,1.178433
4,20000,2863.519644,1.300940
5,50000,2864.885912,1.609051
6,100000,2834.216356,2.215907
